In [ ]:
from pathlib import Path
import shutil
import pandas as pd

import teehr
from teehr import RemoteReadWriteEvaluation
from teehr.evaluation.spark_session_utils import create_spark_session

### Setup
- ready to run for a prefect run with end_WY=2025 and start_WY=2024

In [ ]:
spark = create_spark_session()

In [ ]:
# define evaluation
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

In [ ]:
# define POR for recent test
wy = 2025
end_date = pd.to_datetime(f'09-30-{wy}')

### Check primary

In [ ]:
obs_df = ev.primary_timeseries.filter(
    f"value_time < '{end_date}'"
).to_pandas()

obs_df.head()

In [ ]:
print(obs_df.shape)

In [ ]:
var_list = obs_df['variable_name'].unique()
print(var_list)

### Check Secondary

In [ ]:
fcst_df = ev.secondary_timeseries.filter(
    f"value_time < '{end_date}'"
).to_pandas()
fcst_df.head()

In [ ]:
print(fcst_df.shape)

In [ ]:
var_list = fcst_df['variable_name'].unique()
print(var_list)

### Check single site

#### Fcst

In [ ]:
fcst_df = ev.secondary_timeseries.filter([
    "location_id = 'cbrfc-OAWU1'",
    f"value_time < '{end_date}'",
    "member = 'crx'",
]).order_by(
    'value_time'
).to_pandas()

fcst_df.tail(10)

In [ ]:
print(fcst_df.shape)

In [ ]:
var_list = fcst_df['variable_name'].unique()
print(var_list)

#### Obs

In [ ]:
obs_df = ev.primary_timeseries.filter([
    "location_id = 'cbrfc-OAWU1'",
    f"value_time < '{end_date}'"
]).order_by(
    'value_time'
).to_pandas()

obs_df.tail(10)

In [ ]:
print(obs_df.shape)

In [ ]:
obs_var_list = obs_df['variable_name'].unique()
print(obs_var_list)

### Kill spark

In [ ]:
ev.spark.stop()